Install Dependencies

In [1]:
# ===============================
# Step 1: Install Required Packages
# ===============================
!pip install -q pandas numpy scikit-learn imbalanced-learn lightgbm joblib


Import Libraries

In [2]:
# ===============================
# Block 2: Import Libraries
# ===============================
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from lightgbm import LGBMClassifier, log_evaluation
import joblib


Load Dataset from GitHub

In [3]:
# ===============================
# Block 3: Load Dataset
# ===============================
url = "financial_loan.csv"
df = pd.read_csv(url)
print(" Data loaded:", df.shape)
df.head()


 Data loaded: (38576, 24)


,id,address_state,application_type,emp_length,emp_title,grade,home_ownership,issue_date,last_credit_pull_date,last_payment_date,...,sub_grade,term,verification_status,annual_income,dti,installment,int_rate,loan_amount,total_acc,total_payment
0,1077430,GA,INDIVIDUAL,< 1 year,Ryder,C,RENT,11-02-2021,13-09-2021,13-04-2021,...,C4,60 months,Source Verified,30000.0,0.0100,59.83,0.1527,2500,4,1009
1,1072053,CA,INDIVIDUAL,9 years,MKC Accounting,E,RENT,01-01-2021,14-12-2021,15-01-2021,...,E1,36 months,Source Verified,48000.0,0.0535,109.43,0.1864,3000,4,3939
2,1069243,CA,INDIVIDUAL,4 years,Chemat Technology Inc,C,RENT,05-01-2021,12-12-2021,09-01-2021,...,C5,36 months,Not Verified,50000.0,0.2088,421.65,0.1596,12000,11,3522
3,1041756,TX,INDIVIDUAL,< 1 year,barnes distribution,B,MORTGAGE,25-02-2021,12-12-2021,12-03-2021,...,B2,60 months,Source Verified,42000.0,0.0540,97.06,0.1065,4500,9,4911
4,1068350,IL,INDIVIDUAL,10+ years,J&J Steel Inc,A,MORTGAGE,01-01-2021,14-12-2021,15-01-2021,...,A1,36 months,Verified,83000.0,0.0231,106.53,0.0603,3500,28,3835


Target Mapping & Cleaning

In [4]:
# ===============================
# Block 4: Target Mapping & Cleaning
# ===============================
def map_target(status):
    mapping = {
        "Fully Paid": 1,
        "Current": 1,
        "Charged Off": 0,
        "Default": 0,
        "Late (31-120 days)": 0,
        "Late (16-30 days)": 0,
        "In Grace Period": 0
    }
    return mapping.get(status, np.nan)

df['loan_status_binary'] = df['loan_status'].map(map_target)
df = df.dropna(subset=['loan_status_binary'])
print("Target mapping completed. Sample:")
df[['loan_status', 'loan_status_binary']].head()


Target mapping completed. Sample:


,loan_status,loan_status_binary
0,Charged Off,0
1,Fully Paid,1
2,Charged Off,0
3,Fully Paid,1
4,Fully Paid,1


Feature Engineering

In [5]:
# ===============================
# Block 5: Feature Engineering
# ===============================
def parse_emp_length(val):
    if pd.isna(val): return np.nan
    s = str(val).lower()
    if "10+" in s: return 10
    if "<" in s: return 0
    m = re.search(r"(\d+)", s)
    return float(m.group(1)) if m else np.nan

def parse_term(val):
    if pd.isna(val): return np.nan
    m = re.search(r"(\d+)", str(val))
    return float(m.group(1)) if m else np.nan

def parse_percentage(val):
    if pd.isna(val): return np.nan
    try:
        return float(str(val).replace("%", "").strip())
    except:
        return np.nan

if 'emp_length' in df.columns:
    df['emp_length_num'] = df['emp_length'].map(parse_emp_length)
if 'term' in df.columns:
    df['term_num'] = df['term'].map(parse_term)
if 'int_rate' in df.columns:
    df['int_rate_num'] = df['int_rate'].map(parse_percentage)
if 'revol_util' in df.columns:
    df['revol_util_num'] = df['revol_util'].map(parse_percentage)

df['loan_to_income_ratio'] = df['loan_amount'] / (df['annual_income'] + 1)
df['installment_to_income_ratio'] = df['installment'] / (df['annual_income'] + 1)

if 'grade' in df.columns:
    df['grade_factor'] = pd.factorize(df['grade'])[0]
    df['grade_x_int_rate'] = df['grade_factor'] * df['int_rate_num']

df['term_x_loan_to_income'] = df['term_num'] * df['loan_to_income_ratio']
df['installment_x_dti'] = df['installment_to_income_ratio'] * df['dti']

# Fill missing numerical columns with median
num_cols = [
    'annual_income', 'loan_amount', 'int_rate_num', 'dti', 'emp_length_num',
    'installment', 'total_payment', 'total_acc', 'loan_to_income_ratio',
    'installment_to_income_ratio', 'term_num', 'grade_x_int_rate',
    'term_x_loan_to_income', 'installment_x_dti'
]
num_cols = [col for col in num_cols if col in df.columns]
for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

# Encode categorical variables
cat_cols = ['grade', 'home_ownership', 'purpose', 'verification_status']
cat_cols = [col for col in cat_cols if col in df.columns]
for col in cat_cols:
    df[col] = pd.factorize(df[col])[0]

print("✅ Feature engineering completed. Sample:")
df.head()


✅ Feature engineering completed. Sample:


,id,address_state,application_type,emp_length,emp_title,grade,home_ownership,issue_date,last_credit_pull_date,last_payment_date,...,loan_status_binary,emp_length_num,term_num,int_rate_num,loan_to_income_ratio,installment_to_income_ratio,grade_factor,grade_x_int_rate,term_x_loan_to_income,installment_x_dti
0,1077430,GA,INDIVIDUAL,< 1 year,Ryder,0,0,11-02-2021,13-09-2021,13-04-2021,...,0,0.0,60.0,0.1527,0.083331,0.001994,0,0.0000,4.999833,0.000020
1,1072053,CA,INDIVIDUAL,9 years,MKC Accounting,1,0,01-01-2021,14-12-2021,15-01-2021,...,1,9.0,36.0,0.1864,0.062499,0.002280,1,0.1864,2.249953,0.000122
2,1069243,CA,INDIVIDUAL,4 years,Chemat Technology Inc,0,0,05-01-2021,12-12-2021,09-01-2021,...,0,4.0,36.0,0.1596,0.239995,0.008433,0,0.0000,8.639827,0.001761
3,1041756,TX,INDIVIDUAL,< 1 year,barnes distribution,2,1,25-02-2021,12-12-2021,12-03-2021,...,1,0.0,60.0,0.1065,0.107140,0.002311,2,0.2130,6.428418,0.000125
4,1068350,IL,INDIVIDUAL,10+ years,J&J Steel Inc,3,1,01-01-2021,14-12-2021,15-01-2021,...,1,10.0,36.0,0.0603,0.042168,0.001283,3,0.1809,1.518054,0.000030


Feature Selection & Target

In [6]:
# ===============================
# Block 6: Feature Selection & Target
# ===============================
features = [
    'total_payment', 'installment', 'loan_amount', 'int_rate_num',
    'annual_income', 'dti', 'total_acc', 'term_num',
    'loan_to_income_ratio', 'installment_to_income_ratio',
    'grade_x_int_rate', 'term_x_loan_to_income', 'installment_x_dti'
]
features = [f for f in features if f in df.columns]

X = df[features]
y = df['loan_status_binary'].astype(int)

print(f" Features used: {features}")


 Features used: ['total_payment', 'installment', 'loan_amount', 'int_rate_num', 'annual_income', 'dti', 'total_acc', 'term_num', 'loan_to_income_ratio', 'installment_to_income_ratio', 'grade_x_int_rate', 'term_x_loan_to_income', 'installment_x_dti']


Train/Validation Split

In [7]:
# ===============================
# Block 7: Train/Validation Split
# ===============================
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(" Train shape:", X_train.shape, "| Validation shape:", X_val.shape)


 Train shape: (30860, 13) | Validation shape: (7716, 13)


Handle Class Imbalance (SMOTE)

In [8]:
# ===============================
# Block 8: SMOTE Oversampling
# ===============================
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print(" After SMOTE, training set shape:", X_train_res.shape)


 After SMOTE, training set shape: (53188, 13)


Train LightGBM Model

In [9]:
# ===============================
# Block 9: Train LightGBM
# ===============================
model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    num_leaves=31,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train_res, y_train_res,
    eval_set=[(X_val, y_val)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(50), log_evaluation(50)]
)
print(" Model training completed.")


[LightGBM] [Info] Number of positive: 26594, number of negative: 26594
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007681 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3131
[LightGBM] [Info] Number of data points in the train set: 53188, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
Training until validation scores don't improve for 50 rounds
[50]	valid_0's auc: 0.971486	valid_0's binary_logloss: 0.270867
[100]	valid_0's auc: 0.976794	valid_0's binary_logloss: 0.158834
[150]	valid_0's auc: 0.978231	valid_0's binary_logloss: 0.124761
[200]	valid_0's auc: 0.979274	valid_0's binary_logloss: 0.108997
[250]	valid_0's auc: 0.979646	valid_0's binary_logloss: 0.101739
[300]	valid_0's auc: 0.980123	valid_0's binary_logloss: 0.0979082
[350]	valid_0's auc: 0.980544	valid_0's binary_logloss: 0.0950732
[400]	valid_0's auc: 0.980478	valid_0's bina

Evaluate Model

In [10]:
# ===============================
# Block 10: Evaluate Model
# ===============================
y_proba = model.predict_proba(X_val)[:, 1]
threshold = 0.4
y_pred = (y_proba >= threshold).astype(int)

print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred, digits=4))
print("ROC AUC:", round(roc_auc_score(y_val, y_proba), 4))



Confusion Matrix:
 [[ 896  171]
 [  36 6613]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9614    0.8397    0.8964      1067
           1     0.9748    0.9946    0.9846      6649

    accuracy                         0.9732      7716
   macro avg     0.9681    0.9172    0.9405      7716
weighted avg     0.9729    0.9732    0.9724      7716

ROC AUC: 0.9807


Save Model

In [11]:
# ===============================
# Block 11: Save Model
# ===============================
joblib.dump(model, "EcoCred_model.pkl")
print(" Model saved as EcoCred_model.pkl")


 Model saved as EcoCred_model.pkl
